In [ ]:
import os
import pandas as pd
from pathlib import Path

# Obtiene la ruta actual y obtiene la ruta de la libreria
current_path = Path.cwd()
URBANTRIPS_PATH = current_path.parent
os.chdir(URBANTRIPS_PATH)

from urbantrips.utils import utils
from urbantrips.kpi.kpi import compute_route_section_load
from urbantrips.viz.viz import visualize_route_section_load, viz_etapas_x_tramo_recorrido 
from urbantrips.kpi.line_od_matrix import compute_lines_od_matrix
from urbantrips.viz.line_od_matrix import visualize_lines_od_matrix,viz_line_od_matrix,map_desire_lines
from urbantrips.kpi.supply_kpi import compute_route_section_supply
from urbantrips.viz.section_supply import visualize_route_section_supply_data

from urbantrips.storage.context import StorageContext
from urbantrips.storage.adapters.duckdb.data import DuckDBDataAdapter
from urbantrips.storage.adapters.duckdb.insumos import DuckDBInsumoAdapter
from urbantrips.storage.adapters.duckdb.dash import DuckDBDashAdapter
from urbantrips.storage.adapters.duckdb.general import DuckDBGeneralAdapter
from urbantrips.utils.utils import get_db_path

ctx = StorageContext(
    data=DuckDBDataAdapter(get_db_path("data")),
    insumos=DuckDBInsumoAdapter(get_db_path("insumos")),
    dash=DuckDBDashAdapter(get_db_path("dash")),
    general=DuckDBGeneralAdapter(get_db_path("general")),
)

# Leer archivos de configuración y conexiones a las db
configs = utils.leer_configs_generales()
conn_insumos = utils.iniciar_conexion_db(tipo='insumos')

In [ ]:
# Se leen los datos de las lineas
metadata_lineas = pd.read_sql("select id_linea,nombre_linea, modo from metadata_lineas;", conn_insumos)
# Se puede buscar por nombre de linea que contenga alguna palabra o numero
metadata_lineas[metadata_lineas.nombre_linea.str.contains("100")] #reemplazar 50 por lo que se desee


In [ ]:
rango = [9,9] # Se establece un rango horario, en este caso de 7 a 10 
line_ids = [100] # Se establecen los ids de las lineas a analizar
day_type = 'weekday' # Se establece el tipo de día a analizar puede ser weekday, weekend o una fecha 1/2/2024
section_meters = None # Se establece el parámetro de metros de sección
n_sections = 13 # Se establece el número de secciones a analizar, si se usan metro no se necesita

# Carga por tramo

In [ ]:
# Se calculan los estadisticos de carga de las secciones de las lineas
section_load_df = compute_route_section_load(
    ctx = ctx,
    line_ids=line_ids, hour_range=rango,n_sections=n_sections,
    section_meters = section_meters,day_type=day_type)

section_load_df.head()

In [ ]:
# Se visualizan los estadisticos de carga de las secciones de las lineas
gdf_d0, gdf_d1 = viz_etapas_x_tramo_recorrido(
            ctx,
            section_load_df,
            stat='totals',
            factor=500,
            factor_min=10,
            return_gdfs=True,
            save_gdf=False,
        )
gdf_d0.head()

In [ ]:
gdf_d0.explore(column = 'prop', cmap='viridis')

#  Matriz od y lineas de deseo

In [ ]:
# Se computa la matriz OD de las lineas
line_od_matrix = compute_lines_od_matrix(
    line_ids=line_ids, hour_range=rango,n_sections=n_sections,
    section_meters=section_meters, day_type=day_type, save_csv=True, ctx=ctx
)
line_od_matrix.head()

In [ ]:
# Se visualiza la matriz OD de las lineas
visualize_lines_od_matrix(
    line_ids=line_ids, hour_range=rango,
    day_type=day_type, n_sections=n_sections,section_meters=section_meters,
      stat='totals', ctx=ctx)

In [ ]:
# Calcula los estadisticos de oferta por sección de las lineas
section_meters = 2000 # Se establece el parámetro de metros de sección mas grandes para que haya menos ruido en los datos

route_section_supply = compute_route_section_supply(
    ctx=ctx,
    line_ids=line_ids,
    hour_range=rango,
    section_meters = section_meters,
    day_type=day_type)

In [ ]:
visualize_route_section_supply_data(
    ctx=ctx,
    line_ids=line_ids,
    hour_range=rango,
    day_type="weekday",
    section_meters=section_meters)

In [ ]:
# con este codigo se puede consultar la ayuda de las funciones
visualize_lines_od_matrix?